# 1. SQL in R

In this section, we will use R and SQL to join relevant tables and do Feature Engineering to create relevant new columns.

In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
install.packages(c("sqldf", "ggridges"))
library(sqldf)
library(ggridges)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’

trying URL 'https://cran.rstudio.com/src/contrib/gsubfn_0.7.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/proto_1.0.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RSQLite_2.4.6.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/chron_2.3-62.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/ggridges_0.5.7.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmpe7TBW0/downloaded_packages’
Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite
In addition: Warning message:
no DISPLAY variable so Tk is not available 


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# SQL in R — Database Overview & Data Quality Summary
%%R

library(sqldf)

# Load all cleaned CSVs
app_events <- read.csv('/content/drive/MyDrive/DBA/Initial Files/app_events.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
complaints <- read.csv('/content/drive/MyDrive/DBA/Initial Files/complaints.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
customers  <- read.csv('/content/drive/MyDrive/DBA/Initial Files/customers.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
deliveries <- read.csv('/content/drive/MyDrive/DBA/Initial Files/deliveries.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
drivers    <- read.csv('/content/drive/MyDrive/DBA/Initial Files/drivers.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
hubs       <- read.csv('/content/drive/MyDrive/DBA/Initial Files/hubs.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
incidents  <- read.csv('/content/drive/MyDrive/DBA/Initial Files/incidents.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
orders     <- read.csv('/content/drive/MyDrive/DBA/Initial Files/orders.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))
vehicles   <- read.csv('/content/drive/MyDrive/DBA/Initial Files/vehicles.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

# Generate summary for each table
table_summary <- function(df, table_name) {
  cat("\n")
  cat(rep("=", 60), "\n", sep = "")
  cat("TABLE:", table_name, "\n")
  cat(rep("=", 60), "\n", sep = "")
  cat("Total Rows   :", nrow(df), "\n")
  cat("Total Columns:", ncol(df), "\n")
  cat(rep("-", 60), "\n", sep = "")
  cat(sprintf("%-30s %-10s %-10s\n", "Column", "Type", "Null Count"))
  cat(rep("-", 60), "\n", sep = "")

  for (col in colnames(df)) {
    col_type  <- class(df[[col]])
    null_count <- sum(is.na(df[[col]]))
    cat(sprintf("%-30s %-10s %-10s\n", col, col_type, null_count))
  }
  cat(rep("-", 60), "\n", sep = "")
}
table_summary(deliveries, "deliveries")
table_summary(orders,     "orders")
table_summary(customers,  "customers")
table_summary(complaints, "complaints")
table_summary(drivers,    "drivers")
table_summary(vehicles,   "vehicles")
table_summary(hubs,       "hubs")
table_summary(incidents,  "incidents")
table_summary(app_events, "app_events")

# Complete overview of all the tables
cat("\n")
cat(rep("=", 60), "\n", sep = "")
cat("MASTER OVERVIEW — ALL TABLES\n")
cat(rep("=", 60), "\n", sep = "")
cat(sprintf("%-20s %-10s %-10s %-15s\n", "Table", "Rows", "Columns", "Total Nulls"))
cat(rep("-", 60), "\n", sep = "")

table_list <- list(
  deliveries  = deliveries,
  orders      = orders,
  customers   = customers,
  complaints  = complaints,
  drivers     = drivers,
  vehicles    = vehicles,
  hubs        = hubs,
  incidents   = incidents,
  app_events  = app_events
)

for (name in names(table_list)) {
  df          <- table_list[[name]]
  total_nulls <- sum(is.na(df))
  cat(sprintf("%-20s %-10s %-10s %-15s\n",
              name, nrow(df), ncol(df), total_nulls))
}
cat(rep("=", 60), "\n", sep = "")


TABLE: deliveries 
Total Rows   : 950 
Total Columns: 13 
------------------------------------------------------------
Column                         Type       Null Count
------------------------------------------------------------
delivery_id                    character  0         
order_id                       character  0         
driver_id                      character  0         
vehicle_id                     character  0         
hub_id                         character  0         
dispatch_time                  character  0         
delivery_completed_at          character  19        
delivery_status                character  0         
route_distance_km              numeric    0         
manual_route_override_count    integer    0         
proof_of_completion_missing    integer    0         
customer_rating_post_delivery  numeric    14        
fuel_or_charge_cost            numeric    0         
------------------------------------------------------------

TABLE: orders 


In [ ]:
%%R

# Normalize Zone in all tables
convert <- function(value) {
  value <- trimws(as.character(value))

  # Handle central abbreviations first
  if (tolower(value) %in% c('central', 'ctr')) {
    return('Central')
  }
  # Title case everything else (e.g., "NORTH" -> "North", "AIRPORT" -> "Airport")
  return(tools::toTitleCase(tolower(value)))
}

# Vectorize so it works on entire columns at once
convert_vec <- Vectorize(convert)


# Apply to all tables that have zone-related columns and count how many were changed

count_changes <- function(original, cleaned) {
  sum(original != cleaned, na.rm = TRUE)
}

# deliveries (no zone column directly, skip)

# orders: pickup_zone, dropoff_zone
orders_original_pickup  <- orders$pickup_zone
orders_original_dropoff <- orders$dropoff_zone

orders$pickup_zone  <- convert_vec(orders$pickup_zone)
orders$dropoff_zone <- convert_vec(orders$dropoff_zone)

orders_pickup_changed  <- count_changes(orders_original_pickup,  orders$pickup_zone)
orders_dropoff_changed <- count_changes(orders_original_dropoff, orders$dropoff_zone)

cat("orders — pickup_zone rows normalized  :", orders_pickup_changed,  "\n")
cat("orders — dropoff_zone rows normalized :", orders_dropoff_changed, "\n")


# customers: home_zone
customers_original_zone <- customers$home_zone
customers$home_zone     <- convert_vec(customers$home_zone)
customers_zone_changed  <- count_changes(customers_original_zone, customers$home_zone)

cat("customers — home_zone rows normalized :", customers_zone_changed, "\n")


# drivers: base_zone
drivers_original_zone <- drivers$base_zone
drivers$base_zone     <- convert_vec(drivers$base_zone)
drivers_zone_changed  <- count_changes(drivers_original_zone, drivers$base_zone)

cat("drivers — base_zone rows normalized   :", drivers_zone_changed, "\n")


# vehicles: assigned_zone
vehicles_original_zone <- vehicles$assigned_zone
vehicles$assigned_zone <- convert_vec(vehicles$assigned_zone)
vehicles_zone_changed  <- count_changes(vehicles_original_zone, vehicles$assigned_zone)

cat("vehicles — assigned_zone rows normalized :", vehicles_zone_changed, "\n")


# hubs: zone
hubs_original_zone <- hubs$zone
hubs$zone          <- convert_vec(hubs$zone)
hubs_zone_changed  <- count_changes(hubs_original_zone, hubs$zone)

cat("hubs — zone rows normalized           :", hubs_zone_changed, "\n")


# app_events: zone_context
app_original_zone      <- app_events$zone_context
app_events$zone_context <- convert_vec(app_events$zone_context)
app_zone_changed       <- count_changes(app_original_zone, app_events$zone_context)

cat("app_events — zone_context rows normalized :", app_zone_changed, "\n")


# Complete summary
cat("\n")
cat(rep("=", 50), "\n", sep = "")
cat("ZONE NORMALIZATION SUMMARY\n")
cat(rep("=", 50), "\n", sep = "")
cat(sprintf("%-30s %-10s\n", "Table / Column", "Rows Cleaned"))
cat(rep("-", 50), "\n", sep = "")
cat(sprintf("%-30s %-10s\n", "orders / pickup_zone",     orders_pickup_changed))
cat(sprintf("%-30s %-10s\n", "orders / dropoff_zone",    orders_dropoff_changed))
cat(sprintf("%-30s %-10s\n", "customers / home_zone",    customers_zone_changed))
cat(sprintf("%-30s %-10s\n", "drivers / base_zone",      drivers_zone_changed))
cat(sprintf("%-30s %-10s\n", "vehicles / assigned_zone", vehicles_zone_changed))
cat(sprintf("%-30s %-10s\n", "hubs / zone",              hubs_zone_changed))
cat(sprintf("%-30s %-10s\n", "app_events / zone_context",app_zone_changed))
cat(rep("-", 50), "\n", sep = "")

total_cleaned <- orders_pickup_changed + orders_dropoff_changed +
                 customers_zone_changed + drivers_zone_changed +
                 vehicles_zone_changed + hubs_zone_changed +
                 app_zone_changed

cat(sprintf("%-30s %-10s\n", "TOTAL ROWS NORMALIZED", total_cleaned))
cat(rep("=", 50), "\n", sep = "")


orders — pickup_zone rows normalized  : 693 
orders — dropoff_zone rows normalized : 705 
customers — home_zone rows normalized : 368 
drivers — base_zone rows normalized   : 84 
vehicles — assigned_zone rows normalized : 69 
hubs — zone rows normalized           : 0 
app_events — zone_context rows normalized : 370 

ZONE NORMALIZATION SUMMARY
Table / Column                 Rows Cleaned
--------------------------------------------------
orders / pickup_zone           693       
orders / dropoff_zone          705       
customers / home_zone          368       
drivers / base_zone            84        
vehicles / assigned_zone       69        
hubs / zone                    0         
app_events / zone_context      370       
--------------------------------------------------
TOTAL ROWS NORMALIZED          2289      


# TABLE 1 — hub_performance

In [ ]:
%%R
# Joining Deliveries and Hubs table together based on hub_id
# Calculating total Failed, Delayed, OnTime deliveries
# Calculating Average Failed, Delayed, OnTime deliveries

hub_performance <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    h.zone,
    h.hub_type,
    h.capacity_score,
    COUNT(d.delivery_id)                                                     AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)           AS total_failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)           AS total_delayed,
    SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END)           AS total_ontime,
    ROUND(AVG(CASE WHEN d.delivery_status = 'Failed'  THEN 1.0 ELSE 0.0 END) * 100, 2) AS failure_rate_pct,
    ROUND(AVG(CASE WHEN d.delivery_status = 'Delayed' THEN 1.0 ELSE 0.0 END) * 100, 2) AS delay_rate_pct,
    ROUND(AVG(d.route_distance_km), 2)                                       AS avg_route_distance_km,
    ROUND(AVG(d.manual_route_override_count), 2)                             AS avg_manual_overrides
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_id, h.hub_name, h.zone, h.hub_type, h.capacity_score
  ORDER BY failure_rate_pct DESC
")

print(hub_performance)

  hub_id       hub_name      zone  hub_type capacity_score total_deliveries
1    H08  Midtown Relay   Central  Charging             63              128
2    H05   Central Core   Central   Control             88              115
3    H06    Airport Hub   Airport  Dispatch             71              104
4    H04      West Gate      West  Dispatch             69              127
5    H01 North Exchange     North  Dispatch             82              136
6    H07  Riverside Hub Riverside Warehouse             66              115
7    H02     South Link     South  Dispatch             78              106
8    H03      East Dock      East Warehouse             74              119
  total_failed total_delayed total_ontime failure_rate_pct delay_rate_pct
1           26            22           80            20.31          17.19
2           23            25           67            20.00          21.74
3           15            27           62            14.42          25.96
4           16      

In [ ]:
%%R
# save the table
write.csv(hub_performance, '/content/drive/MyDrive/DBA/SQL_Files/hub_performance.csv', row.names=FALSE)

## Query 1.1

In [ ]:
%%R
hub_performance <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/hub_performance.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

# Get failure and delay and order by in descending to get the most underperforming hubs
query1 <- sqldf("
  SELECT hub_id, hub_name, failure_rate_pct, delay_rate_pct
  FROM hub_performance
  ORDER BY failure_rate_pct DESC, delay_rate_pct DESC
")

# save query
write.csv(query1, '/content/drive/MyDrive/DBA/SQL_Queries/query1.csv', row.names=FALSE)

print(query1)

  hub_id       hub_name failure_rate_pct delay_rate_pct
1    H08  Midtown Relay            20.31          17.19
2    H05   Central Core            20.00          21.74
3    H06    Airport Hub            14.42          25.96
4    H04      West Gate            12.60          22.05
5    H01 North Exchange            12.50          19.12
6    H07  Riverside Hub            12.17          21.74
7    H02     South Link             9.43          24.53
8    H03      East Dock             9.24          19.33


# TABLE 2 — route_override_analysis

In [ ]:
%%R
# Joining Deliveries and Hubs table together base on hub_id
# Getting key stats for manual override
# - total deliveries of manual override
# - avg override per delivery
# - avg distance in km and avg ride cost

route_override_analysis <- sqldf("
  SELECT
    h.hub_name,
    h.zone,
    d.delivery_status,
    COUNT(d.delivery_id)                        AS total_deliveries,
    SUM(d.manual_route_override_count)          AS total_overrides,
    ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides_per_delivery,
    ROUND(AVG(d.route_distance_km), 2)          AS avg_route_km,
    ROUND(AVG(d.fuel_or_charge_cost), 2)        AS avg_cost
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  WHERE d.manual_route_override_count > 0
  GROUP BY h.hub_name, h.zone, d.delivery_status
  ORDER BY total_overrides DESC
")

print(route_override_analysis)

         hub_name      zone delivery_status total_deliveries total_overrides
1  North Exchange     North          OnTime               55              92
2   Midtown Relay   Central          OnTime               51              91
3       East Dock      East          OnTime               44              72
4   Riverside Hub Riverside          OnTime               38              67
5       West Gate      West          OnTime               41              65
6    Central Core   Central          OnTime               41              64
7      South Link     South          OnTime               37              59
8     Airport Hub   Airport          OnTime               34              57
9   Riverside Hub Riverside         Delayed               16              39
10      West Gate      West         Delayed               18              30
11  Midtown Relay   Central          Failed               18              29
12 North Exchange     North         Delayed               17              29

In [ ]:
%%R
# save the table
write.csv(route_override_analysis, '/content/drive/MyDrive/DBA/SQL_Files/route_override_analysis.csv', row.names=FALSE)

## Query 2.1

In [ ]:
%%R
route_override_analysis <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/route_override_analysis.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

query2 <- sqldf("
  SELECT hub_name, delivery_status, total_deliveries, total_overrides, avg_overrides_per_delivery, avg_route_km, avg_cost
  FROM route_override_analysis
  WHERE delivery_status IN ('Delayed', 'Failed')
  ORDER BY hub_name, total_overrides DESC
")

write.csv(query2, '/content/drive/MyDrive/DBA/SQL_Queries/query2.csv', row.names=FALSE)
print(query2)

         hub_name delivery_status total_deliveries total_overrides
1     Airport Hub         Delayed               16              25
2     Airport Hub          Failed               10              13
3    Central Core          Failed               13              24
4    Central Core         Delayed               11              21
5       East Dock         Delayed               12              22
6       East Dock          Failed                7              12
7   Midtown Relay          Failed               18              29
8   Midtown Relay         Delayed               15              22
9  North Exchange         Delayed               17              29
10 North Exchange          Failed               11              19
11  Riverside Hub         Delayed               16              39
12  Riverside Hub          Failed                9              15
13     South Link         Delayed               19              29
14     South Link          Failed                6            

# TABLE 3 — driver_zone_performance

In [ ]:
%%R
# Connecting driver and hub table using driver_id and hub_id
# Compare stats of inzone and out-of-zone drivers

driver_zone_performance <- sqldf("
  SELECT
    dr.base_zone,
    h.zone                                                                    AS operating_zone,
    CASE WHEN dr.base_zone = h.zone THEN 'In-Zone' ELSE 'Out-of-Zone' END    AS zone_match,
    COUNT(d.delivery_id)                                                      AS total_deliveries,
    ROUND(AVG(CASE WHEN d.delivery_status = 'Failed'  THEN 1.0 ELSE 0.0 END) * 100, 2) AS failure_rate_pct,
    ROUND(AVG(CASE WHEN d.delivery_status = 'Delayed' THEN 1.0 ELSE 0.0 END) * 100, 2) AS delay_rate_pct,
    ROUND(AVG(d.manual_route_override_count), 2)                              AS avg_overrides,
    ROUND(AVG(dr.driver_rating), 2)                                           AS avg_driver_rating
  FROM deliveries d
  JOIN drivers  dr ON d.driver_id = dr.driver_id
  JOIN hubs      h ON d.hub_id    = h.hub_id
  GROUP BY dr.base_zone, h.zone, zone_match
  ORDER BY failure_rate_pct DESC
")

print(driver_zone_performance)

   base_zone operating_zone  zone_match total_deliveries failure_rate_pct
1    Airport           West Out-of-Zone               13            30.77
2       West        Central Out-of-Zone               26            30.77
3       West      Riverside Out-of-Zone               13            30.77
4  Riverside        Central Out-of-Zone               23            26.09
5       West          South Out-of-Zone               12            25.00
6  Riverside          South Out-of-Zone                9            22.22
7      South        Central Out-of-Zone               45            22.22
8       East        Central Out-of-Zone               23            21.74
9      North        Airport Out-of-Zone               19            21.05
10   Airport        Central Out-of-Zone               24            20.83
11   Airport          South Out-of-Zone               10            20.00
12   Central        Airport Out-of-Zone               20            20.00
13      West          North Out-of-Zon

In [ ]:
%%R
# save the table
write.csv(driver_zone_performance, '/content/drive/MyDrive/DBA/SQL_Files/driver_zone_performance.csv', row.names=FALSE)

## Query 3.1

In [ ]:
%%R
driver_zone_performance <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/driver_zone_performance.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

query3 <- sqldf("
  SELECT zone_match,
         SUM(total_deliveries)                     AS total_deliveries,
         ROUND(AVG(failure_rate_pct), 2)           AS avg_failure_rate,
         ROUND(AVG(delay_rate_pct), 2)             AS avg_delay_rate,
         ROUND(AVG(avg_overrides), 2)              AS avg_overrides
  FROM driver_zone_performance
  GROUP BY zone_match
  ORDER BY avg_failure_rate DESC
")

write.csv(query3, '/content/drive/MyDrive/DBA/SQL_Queries/query3.csv', row.names=FALSE)
print(query3)

   zone_match total_deliveries avg_failure_rate avg_delay_rate avg_overrides
1 Out-of-Zone              812            13.94          20.06          0.90
2     In-Zone              138            10.30          22.91          1.19


## Query 3.2

In [ ]:
%%R

query32 <- sqldf("
  SELECT base_zone,
         operating_zone,
         total_deliveries,
         failure_rate_pct,
         delay_rate_pct,
         avg_overrides
  FROM driver_zone_performance
  WHERE zone_match = 'Out-of-Zone'
  ORDER BY failure_rate_pct DESC
  LIMIT 10
")

write.csv(query32, '/content/drive/MyDrive/DBA/SQL_Queries/query3.2.csv', row.names=FALSE)
print(query32)

   base_zone operating_zone total_deliveries failure_rate_pct delay_rate_pct
1    Airport           West               13            30.77           7.69
2       West        Central               26            30.77          15.38
3       West      Riverside               13            30.77           7.69
4  Riverside        Central               23            26.09          17.39
5       West          South               12            25.00          16.67
6  Riverside          South                9            22.22           0.00
7      South        Central               45            22.22          17.78
8       East        Central               23            21.74           8.70
9      North        Airport               19            21.05          15.79
10   Airport        Central               24            20.83          25.00
   avg_overrides
1           1.08
2           0.92
3           0.77
4           1.04
5           0.75
6           0.11
7           0.91
8           0.87


## Query 3.3

In [ ]:
%%R

query33 <- sqldf("
  SELECT operating_zone,
         zone_match,
         SUM(total_deliveries)            AS total_deliveries,
         ROUND(AVG(failure_rate_pct), 2)  AS avg_failure_rate,
         ROUND(AVG(avg_overrides), 2)     AS avg_overrides
  FROM driver_zone_performance
  GROUP BY operating_zone
  ORDER BY avg_failure_rate DESC
")

write.csv(query33, '/content/drive/MyDrive/DBA/SQL_Queries/query3.3.csv', row.names=FALSE)
print(query33)

  operating_zone  zone_match total_deliveries avg_failure_rate avg_overrides
1        Central Out-of-Zone              243            21.64          1.00
2           West Out-of-Zone              127            13.63          0.88
3        Airport Out-of-Zone              104            13.26          0.91
4      Riverside Out-of-Zone              115            12.95          1.06
5          North Out-of-Zone              136            12.30          0.97
6          South Out-of-Zone              106            11.47          0.92
7           East Out-of-Zone              119             8.72          0.82


# TABLE 4 — customer_failure_view

In [ ]:
%%R
# Joining Deliveries, Complaints, Incidents using order_id, customer_id, delivery_id
# Aggregating service failures to identify high-risk customers needing immediate recovery

customer_failure_view <- sqldf("
  SELECT
    o.customer_id,
    COUNT(DISTINCT o.order_id)                                     AS total_orders,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS failed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_deliveries,
    COUNT(DISTINCT c.complaint_id)                                 AS total_complaints,
    COUNT(DISTINCT i.incident_id)                                  AS total_incidents,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                 AS avg_rating,
    SUM(COALESCE(c.compensation_amount, 0))                        AS total_compensation_paid
  FROM orders o
  LEFT JOIN deliveries d  ON o.order_id    = d.order_id
  LEFT JOIN complaints c  ON o.customer_id = c.customer_id
  LEFT JOIN incidents  i  ON d.delivery_id = i.delivery_id
  GROUP BY o.customer_id
  HAVING failed_deliveries > 0 OR total_complaints > 0
  ORDER BY total_complaints DESC, failed_deliveries DESC
")

print(customer_failure_view)

    customer_id total_orders failed_deliveries delayed_deliveries
1         C0368            3                 4                  0
2         C0110            3                 3                  0
3         C0282            2                 3                  0
4         C0372            6                 3                  3
5         C0573            3                 3                  0
6         C0142            2                 0                  0
7         C0172            4                 0                  0
8         C0191            2                 0                  0
9         C0242            5                 0                  3
10        C0421            3                 0                  0
11        C0545            6                 0                  6
12        C0626            3                 0                  3
13        C0178            2                 4                  2
14        C0004            3                 2                  0
15        

In [ ]:
%%R
# save the table
write.csv(customer_failure_view, '/content/drive/MyDrive/DBA/SQL_Files/customer_failure_view.csv', row.names=FALSE)

## Query 4.1

In [ ]:
%%R
customer_failure_view <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/customer_failure_view.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

query41 <- sqldf("
  SELECT
    CASE
      WHEN failed_deliveries = 1 THEN '1 Failure'
      WHEN failed_deliveries = 2 THEN '2 Failures'
      WHEN failed_deliveries >= 3 THEN '3+ Failures'
    END                                         AS failure_category,
    COUNT(customer_id)                          AS customer_count,
    ROUND(AVG(total_complaints), 2)             AS avg_complaints,
    ROUND(AVG(avg_rating), 2)                   AS avg_rating,
    ROUND(SUM(total_compensation_paid), 2)      AS total_compensation
  FROM customer_failure_view
  WHERE failed_deliveries > 0
  GROUP BY failure_category
  ORDER BY failed_deliveries ASC
")

write.csv(query41, '/content/drive/MyDrive/DBA/SQL_Queries/query4.1.csv', row.names=FALSE)
print(query41)

  failure_category customer_count avg_complaints avg_rating total_compensation
1        1 Failure             97           0.39       3.32            2225.82
2       2 Failures             22           1.64       3.78            2857.98
3      3+ Failures              8           2.25       3.02            1341.88


## Query 4.2

In [ ]:
%%R

query42 <- sqldf("
  SELECT
    CASE WHEN failed_deliveries > 0 AND total_complaints > 0
         THEN 'Failed + Complained'
         WHEN failed_deliveries > 0 AND total_complaints = 0
         THEN 'Failed but No Complaint'
         WHEN failed_deliveries = 0 AND total_complaints > 0
         THEN 'Complained but No Failure'
    END                                         AS customer_segment,
    COUNT(customer_id)                          AS customer_count,
    ROUND(AVG(avg_rating), 2)                   AS avg_rating,
    ROUND(AVG(total_incidents), 2)              AS avg_incidents,
    ROUND(SUM(total_compensation_paid), 2)      AS total_compensation
  FROM customer_failure_view
  GROUP BY customer_segment
  ORDER BY total_compensation DESC
")

write.csv(query42, '/content/drive/MyDrive/DBA/SQL_Queries/query4.2.csv', row.names=FALSE)
print(query42)

           customer_segment customer_count avg_rating avg_incidents
1 Complained but No Failure            170       4.00          0.55
2       Failed + Complained             63       3.53          0.62
3   Failed but No Complaint             64       3.24          0.67
  total_compensation
1           11103.63
2            6425.68
3               0.00


## Query 4.3

In [ ]:
%%R

query43 <- sqldf("
  SELECT
    failed_deliveries,
    total_complaints,
    COUNT(customer_id)                          AS customer_count,
    ROUND(AVG(avg_rating), 2)                   AS avg_rating,
    ROUND(AVG(total_incidents), 2)              AS avg_incidents,
    ROUND(SUM(total_compensation_paid), 2)      AS total_compensation
  FROM customer_failure_view
  WHERE failed_deliveries <= 3
  GROUP BY failed_deliveries, total_complaints
  ORDER BY failed_deliveries ASC, total_complaints ASC
")

write.csv(query43, '/content/drive/MyDrive/DBA/SQL_Queries/query4.3.csv', row.names=FALSE)
print(query43)

   failed_deliveries total_complaints customer_count avg_rating avg_incidents
1                  0                1            119       3.99          0.47
2                  0                2             44       3.97          0.75
3                  0                3              7       4.20          0.57
4                  1                0             59       3.23          0.53
5                  1                1             38       3.46          0.61
6                  2                0              3       3.71          2.33
7                  2                1              2       3.80          1.00
8                  2                2             17       3.80          0.59
9                  3                0              2       2.80          2.50
10                 3                3              4       2.87          0.50
   total_compensation
1             5055.56
2             4198.73
3             1849.34
4                0.00
5             2225.82
6         

# TABLE 5 — complaint_delivery_link

In [ ]:
%%R
# Joining complaints and orders using order_id, and joining orders and deliveries using order_id
# Finds complaint records with delivery logs to identify service failures

complaint_delivery_link <- sqldf("
  SELECT
    c.complaint_id,
    c.customer_id,
    c.complaint_type,
    c.severity,
    c.channel,
    c.status AS complaint_status,
    c.resolution_days,
    c.compensation_amount,
    d.delivery_status,
    d.manual_route_override_count,
    d.proof_of_completion_missing,
    d.customer_rating_post_delivery
  FROM complaints c
  JOIN orders     o ON c.order_id    = o.order_id
  JOIN deliveries d ON o.order_id    = d.order_id
  WHERE c.severity IN ('High', 'Medium')
  ORDER BY c.severity DESC, c.resolution_days DESC
")

print(complaint_delivery_link)

    complaint_id customer_id    complaint_type severity channel
1         CP0037       C0561      MissedPickup   Medium Chatbot
2         CP0098       C0482      MissedPickup   Medium     App
3         CP0258       C0013 SupportExperience   Medium     App
4         CP0011       C0618             Delay   Medium     App
5         CP0097       C0282   DriverBehaviour   Medium     App
6         CP0126       C0093             Delay   Medium     App
7         CP0014       C0545           Billing   Medium Chatbot
8         CP0099       C0110          AppIssue   Medium   Email
9         CP0171       C0020          AppIssue   Medium   Phone
10        CP0268       C0445          AppIssue   Medium     App
11        CP0026       C0408 SupportExperience   Medium     App
12        CP0164       C0062             Delay   Medium Chatbot
13        CP0186       C0391             Delay   Medium     App
14        CP0201       C0168             Delay   Medium Chatbot
15        CP0253       C0567          Ap

In [ ]:
%%R
# save the table
write.csv(complaint_delivery_link, '/content/drive/MyDrive/DBA/SQL_Files/complaint_delivery_link.csv', row.names=FALSE)

## Query 5.1

In [ ]:
%%R
complaint_delivery_link <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/complaint_delivery_link.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

In [ ]:
%%R

query51 <- sqldf("
  SELECT
    severity,
    delivery_status,
    COUNT(complaint_id)                           AS total_complaints,
    ROUND(AVG(resolution_days), 2)                AS avg_resolution_days,
    ROUND(AVG(compensation_amount), 2)            AS avg_compensation,
    ROUND(AVG(customer_rating_post_delivery), 2)  AS avg_rating
  FROM complaint_delivery_link
  GROUP BY severity, delivery_status
  ORDER BY severity DESC, delivery_status ASC
")

write.csv(query51, '/content/drive/MyDrive/DBA/SQL_Queries/query5.1.csv', row.names=FALSE)
print(query51)

  severity delivery_status total_complaints avg_resolution_days
1   Medium         Delayed               30                5.30
2   Medium          Failed               17                7.00
3   Medium          OnTime               86                6.51
4     High         Delayed               12               15.25
5     High          Failed               13               14.38
6     High          OnTime               28               12.36
  avg_compensation avg_rating
1            16.46       3.22
2            20.43       3.47
3            17.13       4.31
4            34.15       3.21
5            39.26       2.99
6            39.13       4.32


## Query 5.2

In [ ]:
%%R

query52 <- sqldf("
  SELECT
    complaint_type,
    delivery_status,
    COUNT(complaint_id)                           AS total_complaints,
    ROUND(AVG(resolution_days), 2)                AS avg_resolution_days,
    ROUND(AVG(compensation_amount), 2)            AS avg_compensation,
    ROUND(AVG(customer_rating_post_delivery), 2)  AS avg_rating,
    SUM(proof_of_completion_missing)              AS missing_proof_count
  FROM complaint_delivery_link
  GROUP BY complaint_type, delivery_status
  ORDER BY avg_resolution_days DESC
")

write.csv(query52, '/content/drive/MyDrive/DBA/SQL_Queries/query5.2.csv', row.names=FALSE)
print(query52)

      complaint_type delivery_status total_complaints avg_resolution_days
1             Damage          Failed                1               21.00
2             Damage         Delayed                1               16.00
3    DriverBehaviour          Failed                4               13.00
4           AppIssue         Delayed                6               11.83
5             Damage          OnTime                4               11.75
6    DriverBehaviour         Delayed                5               11.60
7  SupportExperience          Failed                3               11.33
8           AppIssue          Failed                4               10.25
9              Delay          Failed                9                9.44
10          AppIssue          OnTime               17                9.41
11 SupportExperience         Delayed                2                8.50
12      MissedPickup          Failed                7                8.43
13      MissedPickup          OnTime  

## Query 5.3

In [ ]:
%%R

sqldf("
  SELECT
    delivery_status,
    COUNT(complaint_id)                           AS total_complaints,
    ROUND(AVG(resolution_days), 2)                AS avg_resolution_days,
    ROUND(AVG(compensation_amount), 2)            AS avg_compensation,
    ROUND(AVG(customer_rating_post_delivery), 2)  AS avg_rating,
    SUM(manual_route_override_count)              AS total_overrides
  FROM complaint_delivery_link
  GROUP BY delivery_status
  ORDER BY total_complaints DESC
")

  delivery_status total_complaints avg_resolution_days avg_compensation
1          OnTime              114                7.95            22.13
2         Delayed               42                8.14            20.29
3          Failed               30               10.20            28.59
  avg_rating total_overrides
1       4.31              90
2       3.22              46
3       3.26              32


# TABLE 6 — high_risk_customers

In [ ]:
%%R
# Joining order and deliveries using order_id
# Joining orders and customers using customer_id
# Left join complaints and customers using customer_id
# Identifying high risk accounts by finding customer profiles from their order history
# This query flags high-value customers experiencing repeated service failures

high_risk_customers <- sqldf("
  SELECT
    o.customer_id,
    cu.customer_type,
    cu.home_zone,
    cu.loyalty_score,
    cu.account_status,
    COUNT(DISTINCT o.order_id)                                      AS total_orders,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)  AS failed_deliveries,
    COUNT(DISTINCT c.complaint_id)                                  AS total_complaints,
    SUM(COALESCE(c.compensation_amount, 0))                         AS total_compensation,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                  AS avg_rating
  FROM orders o
  JOIN deliveries d  ON o.order_id    = d.order_id
  JOIN customers cu  ON o.customer_id = cu.customer_id
  LEFT JOIN complaints c ON o.customer_id = c.customer_id
  GROUP BY o.customer_id, cu.customer_type, cu.home_zone,
           cu.loyalty_score, cu.account_status
  HAVING failed_deliveries > 1 AND total_complaints >= 1
  ORDER BY total_compensation DESC, failed_deliveries DESC
")

print(high_risk_customers)


   customer_id customer_type home_zone loyalty_score account_status
1        C0351    Enterprise   Central          48.6         Active
2        C0372      Consumer      West          26.2         Active
3        C0368      Consumer     North          49.5         Active
4        C0023      Consumer     South          73.1         Active
5        C0618      Consumer   Airport          68.4         Active
6        C0186      Consumer      East          57.5        Dormant
7        C0597      Consumer     North          57.2         Active
8        C0282      Consumer Riverside          71.4         Active
9        C0529           SME Riverside          43.2         Active
10       C0339      Consumer     South          46.6         Active
11       C0476      Consumer     North          66.0         Active
12       C0533      Consumer     South          46.9         Active
13       C0480    Enterprise     North          67.6        Dormant
14       C0573           SME   Airport          

In [ ]:
%%R
# save the table
write.csv(high_risk_customers, '/content/drive/MyDrive/DBA/SQL_Files/high_risk_customers.csv', row.names=FALSE)

In [ ]:
%%R
high_risk_customers <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/high_risk_customers.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

## Query 6.1

In [ ]:
%%R

query61 <- sqldf("
  SELECT
    customer_type,
    account_status,
    COUNT(customer_id)                          AS total_customers,
    ROUND(AVG(total_orders), 2)                 AS avg_orders,
    ROUND(AVG(failed_deliveries), 2)            AS avg_failures,
    ROUND(AVG(total_complaints), 2)             AS avg_complaints,
    ROUND(AVG(avg_rating), 2)                   AS avg_rating,
    ROUND(SUM(total_compensation), 2)           AS total_compensation
  FROM high_risk_customers
  GROUP BY customer_type, account_status
  ORDER BY total_compensation DESC
")

write.csv(query61, '/content/drive/MyDrive/DBA/SQL_Queries/query6.1.csv', row.names=FALSE)
print(query61)

  customer_type account_status total_customers avg_orders avg_failures
1      Consumer         Active              17       2.88         2.29
2           SME         Active               3       2.67         2.33
3    Enterprise         Active               1       3.00         2.00
4      Consumer        Dormant               3       2.00         2.00
5    Enterprise        Dormant               1       3.00         2.00
  avg_complaints avg_rating total_compensation
1           2.24       3.73            2154.77
2           2.33       3.05             328.22
3           2.00       3.74             306.06
4           1.67       3.43             303.46
5           2.00       3.91             116.28


## Query 6.2

In [ ]:
%%R

query62 <- sqldf("
  SELECT
    home_zone,
    COUNT(customer_id)                          AS high_risk_customers,
    ROUND(AVG(failed_deliveries), 2)            AS avg_failures,
    ROUND(AVG(total_complaints), 2)             AS avg_complaints,
    ROUND(AVG(loyalty_score), 2)                AS avg_loyalty_score,
    ROUND(AVG(avg_rating), 2)                   AS avg_rating,
    ROUND(SUM(total_compensation), 2)           AS total_compensation
  FROM high_risk_customers
  GROUP BY home_zone
  ORDER BY high_risk_customers DESC
")

write.csv(query62, '/content/drive/MyDrive/DBA/SQL_Queries/query6.2.csv', row.names=FALSE)
print(query62)

  home_zone high_risk_customers avg_failures avg_complaints avg_loyalty_score
1     North                   5         2.40           2.40             62.28
2      East                   4         2.25           2.00             59.90
3   Airport                   4         2.25           2.25             63.03
4      West                   3         2.33           2.33             51.20
5     South                   3         2.00           1.67             55.53
6 Riverside                   3         2.33           2.33             47.50
7   Central                   3         2.00           2.00             51.73
  avg_rating total_compensation
1       3.72             707.26
2       3.33             404.34
3       3.17             459.30
4       3.47             378.91
5       3.90             486.71
6       3.98             327.37
7       3.98             444.90


# TABLE 7 — Service Profitability Proxy by Service Type

In [ ]:
%%R
# Joininig order and deliveries using order_id
# Analyzing profitability across services by calculating margins and cost leakage
# This query helps the Finance Director identify which service tiers are draining the budget

service_profitability <- sqldf("
  SELECT
    o.service_type,
    o.priority_level,
    COUNT(d.delivery_id)                                          AS total_deliveries,
    ROUND(AVG(o.order_value), 2)                                  AS avg_order_value,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                          AS avg_delivery_cost,
    ROUND(AVG(o.order_value - d.fuel_or_charge_cost), 2)          AS avg_gross_margin,
    ROUND(SUM(o.order_value), 2)                                  AS total_revenue,
    ROUND(SUM(d.fuel_or_charge_cost), 2)                          AS total_cost,
    ROUND(SUM(o.order_value - d.fuel_or_charge_cost), 2)          AS total_gross_margin,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_count,
    ROUND(SUM(CASE WHEN d.delivery_status = 'Failed'
              THEN d.fuel_or_charge_cost ELSE 0 END), 2)          AS cost_on_failed_deliveries
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.service_type, o.priority_level
  ORDER BY avg_gross_margin ASC
")

print(service_profitability)

   service_type priority_level total_deliveries avg_order_value
1       Medical       Critical               10           71.76
2       Medical            Low               37           79.91
3        Retail       Critical               18           79.71
4        Retail            Low               57           83.44
5        Parcel       Critical               17           82.10
6        Parcel            Low               62           84.72
7        Retail           High               52           84.45
8        Parcel         Medium               89           88.99
9       Medical           High               21           87.82
10       Retail         Medium               97           91.37
11     Business           High               35           92.77
12    Passenger         Medium              105           93.44
13      Medical         Medium               40           95.66
14     Business            Low               26           95.60
15    Passenger            Low          

In [ ]:
%%R
# save the table
write.csv(service_profitability, '/content/drive/MyDrive/DBA/SQL_Files/service_profitability.csv', row.names=FALSE)

In [ ]:
%%R
service_profitability <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/service_profitability.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

## Query 7.1

In [ ]:
%%R

query71 <- sqldf("
  SELECT
    service_type,
    total_deliveries,
    avg_order_value,
    avg_delivery_cost,
    avg_gross_margin,
    total_revenue,
    total_cost,
    total_gross_margin,
    failed_count,
    cost_on_failed_deliveries
  FROM service_profitability
  GROUP BY service_type
  ORDER BY avg_gross_margin ASC
")

write.csv(query71, '/content/drive/MyDrive/DBA/SQL_Queries/query7.1.csv', row.names=FALSE)
print(query71)

  service_type total_deliveries avg_order_value avg_delivery_cost
1      Medical               10           71.76             11.15
2       Retail               18           79.71             11.71
3       Parcel               17           82.10             11.00
4     Business               35           92.77             13.37
5    Passenger              105           93.44             12.65
  avg_gross_margin total_revenue total_cost total_gross_margin failed_count
1            60.60        717.56     111.54             606.02            1
2            68.00       1434.84     210.75            1224.09            0
3            71.10       1395.78     187.02            1208.76            0
4            79.40       3246.81     467.80            2779.01            6
5            80.79       9811.26    1328.58            8482.68           13
  cost_on_failed_deliveries
1                     18.12
2                      0.00
3                      0.00
4                     77.64
5       

## Query 7.2

In [ ]:
%%R

query72 <- sqldf("
  SELECT
    priority_level,
    COUNT(service_type)                           AS service_segments,
    ROUND(SUM(total_deliveries), 2)               AS total_deliveries,
    ROUND(AVG(avg_order_value), 2)                AS avg_order_value,
    ROUND(AVG(avg_delivery_cost), 2)              AS avg_delivery_cost,
    ROUND(AVG(avg_gross_margin), 2)               AS avg_gross_margin,
    ROUND(SUM(cost_on_failed_deliveries), 2)      AS total_cost_wasted_on_failures
  FROM service_profitability
  GROUP BY priority_level
  ORDER BY avg_gross_margin ASC
")

write.csv(query72, '/content/drive/MyDrive/DBA/SQL_Queries/query7.2.csv', row.names=FALSE)
print(query72)

  priority_level service_segments total_deliveries avg_order_value
1            Low                5              259           87.85
2       Critical                5               74           87.24
3         Medium                5              386           93.99
4           High                5              231           93.82
  avg_delivery_cost avg_gross_margin total_cost_wasted_on_failures
1             12.87            74.98                        417.76
2             11.36            75.88                         43.99
3             13.28            80.71                        864.45
4             12.65            81.17                        409.33


# TABLE 8 — Zone-Level Cost and Loss Analysis

In [ ]:
%%R
# Joining order with deliveries using order_id
# Also joining complaints and order using customer_id
# To find profits by region by calculating net margins after other costs and compensation
# This analysis identifies which geographic zones are currently underperforming financially

zone_cost_analysis <- sqldf("
  SELECT
    o.pickup_zone,
    COUNT(d.delivery_id)                                          AS total_deliveries,
    ROUND(SUM(o.order_value), 2)                                  AS total_revenue,
    ROUND(SUM(d.fuel_or_charge_cost), 2)                          AS total_delivery_cost,
    ROUND(SUM(o.order_value - d.fuel_or_charge_cost), 2)          AS total_gross_margin,
    ROUND(SUM(COALESCE(c.compensation_amount, 0)), 2)             AS total_compensation,
    ROUND(SUM(o.order_value - d.fuel_or_charge_cost)
          - SUM(COALESCE(c.compensation_amount, 0)), 2)           AS net_margin_after_compensation,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                          AS avg_cost_per_delivery
  FROM orders o
  JOIN deliveries  d ON o.order_id    = d.order_id
  LEFT JOIN complaints c ON o.customer_id = c.customer_id
  GROUP BY o.pickup_zone
  ORDER BY net_margin_after_compensation ASC
")

print(zone_cost_analysis)

  pickup_zone total_deliveries total_revenue total_delivery_cost
1   Riverside              145      12711.94             1793.67
2        West              135      13613.09             1622.52
3     Airport              139      14364.36             2344.16
4       North              171      15106.00             2014.12
5       South              164      15731.04             2053.29
6        East              187      17008.99             2315.86
7     Central              216      18902.80             2647.58
  total_gross_margin total_compensation net_margin_after_compensation
1           10918.27            1490.74                       9427.53
2           11990.57            1441.67                      10548.90
3           12020.20            1451.69                      10568.51
4           13091.88            1923.00                      11168.88
5           13677.75            1694.26                      11983.49
6           14693.13            2236.46                     

In [ ]:
%%R
# save the table
write.csv(zone_cost_analysis, '/content/drive/MyDrive/DBA/SQL_Files/zone_cost_analysis.csv', row.names=FALSE)

In [ ]:
%%R
zone_cost_analysis <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/zone_cost_analysis.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

## Query 8.1

In [ ]:
%%R

query81 <- sqldf("
  SELECT
    pickup_zone,
    total_deliveries,
    total_revenue,
    total_delivery_cost,
    total_gross_margin,
    total_compensation,
    net_margin_after_compensation,
    failed_deliveries,
    avg_cost_per_delivery
  FROM zone_cost_analysis
  ORDER BY net_margin_after_compensation ASC
")


write.csv(query81, '/content/drive/MyDrive/DBA/SQL_Queries/query8.1.csv', row.names=FALSE)
print(query81)

  pickup_zone total_deliveries total_revenue total_delivery_cost
1   Riverside              145      12711.94             1793.67
2        West              135      13613.09             1622.52
3     Airport              139      14364.36             2344.16
4       North              171      15106.00             2014.12
5       South              164      15731.04             2053.29
6        East              187      17008.99             2315.86
7     Central              216      18902.80             2647.58
  total_gross_margin total_compensation net_margin_after_compensation
1           10918.27            1490.74                       9427.53
2           11990.57            1441.67                      10548.90
3           12020.20            1451.69                      10568.51
4           13091.88            1923.00                      11168.88
5           13677.75            1694.26                      11983.49
6           14693.13            2236.46                     

## Query 8.2

In [ ]:
%%R

query82 <- sqldf("
  SELECT
    pickup_zone,
    total_deliveries,
    ROUND(total_revenue / total_deliveries, 2)              AS revenue_per_delivery,
    avg_cost_per_delivery,
    ROUND(total_gross_margin / total_deliveries, 2)         AS margin_per_delivery,
    ROUND(total_compensation / total_deliveries, 2)         AS compensation_per_delivery,
    ROUND(net_margin_after_compensation / total_deliveries, 2) AS net_margin_per_delivery,
    failed_deliveries
  FROM zone_cost_analysis
  ORDER BY net_margin_per_delivery ASC
")

write.csv(query82, '/content/drive/MyDrive/DBA/SQL_Queries/query8.2.csv', row.names=FALSE)
print(query82)

  pickup_zone total_deliveries revenue_per_delivery avg_cost_per_delivery
1     Central              216                87.51                 12.26
2   Riverside              145                87.67                 12.37
3       North              171                88.34                 11.78
4        East              187                90.96                 12.38
5       South              164                95.92                 12.52
6     Airport              139               103.34                 16.86
7        West              135               100.84                 12.02
  margin_per_delivery compensation_per_delivery net_margin_per_delivery
1               75.26                     11.94                   63.32
2               75.30                     10.28                   65.02
3               76.56                     11.25                   65.32
4               78.57                     11.96                   66.61
5               83.40                     10.33 

# TABLE 9 — Compensation Drain by Complaint Type

In [ ]:
%%R
# Analyzing the financial costs of various complaint types to prioritize budget allocation
# This query highlights where high-volume 'minor' issues might be outweighing 'critical' outliers

compensation_by_complaint <- sqldf("
  SELECT
    complaint_type,
    severity,
    COUNT(complaint_id)                  AS total_complaints,
    SUM(compensation_amount)             AS total_compensation_paid,
    ROUND(AVG(compensation_amount), 2)   AS avg_compensation,
    MAX(compensation_amount)             AS max_compensation,
    ROUND(AVG(resolution_days), 2)       AS avg_resolution_days
  FROM complaints
  WHERE compensation_amount IS NOT NULL
    AND compensation_amount > 0
  GROUP BY complaint_type, severity
  ORDER BY total_compensation_paid DESC
")

print(compensation_by_complaint)

      complaint_type severity total_complaints total_compensation_paid
1              Delay   Medium               47                  964.87
2       MissedPickup     High               16                  689.11
3       MissedPickup   Medium               34                  644.80
4              Delay     High               14                  511.54
5    DriverBehaviour   Medium               28                  476.29
6    DriverBehaviour     High               12                  460.63
7           AppIssue     High               12                  408.59
8           AppIssue   Medium               22                  386.58
9             Damage     High                7                  260.85
10 SupportExperience   Medium               11                  224.13
11             Delay      Low               19                  220.43
12           Billing     High                4                  209.29
13          AppIssue      Low               12                  185.55
14    

In [ ]:
%%R
# save the table
write.csv(compensation_by_complaint, '/content/drive/MyDrive/DBA/SQL_Files/compensation_by_complaint.csv', row.names=FALSE)

In [ ]:
%%R
compensation_by_complaint <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/compensation_by_complaint.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

## Query 9.1

In [ ]:
%%R

query91 <- sqldf("
  SELECT
    complaint_type,
    SUM(total_complaints)                         AS total_complaints,
    SUM(total_compensation_paid)                  AS total_compensation,
    ROUND(AVG(avg_compensation), 2)               AS avg_compensation,
    MAX(max_compensation)                         AS max_compensation,
    ROUND(AVG(avg_resolution_days), 2)            AS avg_resolution_days
  FROM compensation_by_complaint
  GROUP BY complaint_type
  ORDER BY total_compensation DESC
")

write.csv(query91, '/content/drive/MyDrive/DBA/SQL_Queries/query9.1.csv', row.names=FALSE)
print(query91)

     complaint_type total_complaints total_compensation avg_compensation
1             Delay               80            1696.84            22.89
2      MissedPickup               58            1423.40            24.41
3          AppIssue               46             980.72            22.36
4   DriverBehaviour               44             973.06            21.48
5           Billing               16             381.94            25.36
6            Damage               14             359.73            22.40
7 SupportExperience               17             342.50            19.94
  max_compensation avg_resolution_days
1            59.75                8.70
2            61.85                8.06
3            47.81                8.70
4            61.11                8.61
5            58.46                8.59
6            57.63               10.71
7            46.47                9.25


## Query 9.2

In [ ]:
%%R

query92 <- sqldf("
  SELECT
    complaint_type,
    severity,
    total_complaints,
    total_compensation_paid,
    avg_compensation,
    avg_resolution_days,
    ROUND(total_compensation_paid / avg_resolution_days, 2) AS compensation_per_day
  FROM compensation_by_complaint
  ORDER BY compensation_per_day DESC
  LIMIT 10
")

write.csv(query92, '/content/drive/MyDrive/DBA/SQL_Queries/query9.2.csv', row.names=FALSE)
print(query92)

      complaint_type severity total_complaints total_compensation_paid
1              Delay   Medium               47                  964.87
2       MissedPickup   Medium               34                  644.80
3    DriverBehaviour   Medium               28                  476.29
4       MissedPickup     High               16                  689.11
5           AppIssue   Medium               22                  386.58
6           AppIssue      Low               12                  185.55
7              Delay     High               14                  511.54
8  SupportExperience   Medium               11                  224.13
9    DriverBehaviour     High               12                  460.63
10             Delay      Low               19                  220.43
   avg_compensation avg_resolution_days compensation_per_day
1             20.53                5.96               161.89
2             18.96                6.24               103.33
3             17.01                5

# TABLE 10 — Semi-Structured Data Complexity Assessment


In [ ]:
%%R
# Analyzing application event response time to find technical performance bottlenecks
# This query maps system latency and failure rates across different devices and zones

app_event_complexity <- sqldf("
  SELECT
    event_type,
    device_type,
    zone_context,
    COUNT(event_id)                                                       AS total_events,
    SUM(CASE WHEN order_id IS NULL OR order_id = '' THEN 1 ELSE 0 END)    AS events_without_order,
    ROUND(AVG(api_latency_ms), 2)                                         AS avg_api_latency_ms,
    MAX(api_latency_ms)                                                   AS max_api_latency_ms,
    SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END)                     AS failed_events,
    ROUND(AVG(CASE WHEN success_flag = 0 THEN 1.0 ELSE 0.0 END) * 100, 2) AS failure_rate_pct
  FROM app_events
  GROUP BY event_type, device_type, zone_context
  ORDER BY total_events DESC
")

print(app_event_complexity)

                     event_type device_type zone_context total_events
1                  search_route     Android        South           17
2                   track_order     Android        North           12
3                   track_order     Android         West           12
4   delivery_instruction_update     Android         East           11
5                  search_route         iOS      Airport           11
6                   track_order     Android         East           11
7                   chat_opened     Android      Central           10
8                   eta_refresh     Android      Central           10
9                  search_route     Android      Central           10
10                  chat_opened     Android        North            9
11                  chat_opened     Android         West            9
12  delivery_instruction_update         iOS    Riverside            9
13                  eta_refresh     Android        North            9
14                pa

In [ ]:
%%R
# save the table
write.csv(app_event_complexity, '/content/drive/MyDrive/DBA/SQL_Files/app_event_complexity.csv', row.names=FALSE)

In [ ]:
%%R
app_event_complexity <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/app_event_complexity.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

## Query 10.1

In [ ]:
%%R

query101 <- sqldf("
  SELECT
    event_type,
    SUM(total_events)                             AS total_events,
    SUM(events_without_order)                     AS unlinked_events,
    ROUND(AVG(avg_api_latency_ms), 2)             AS avg_latency_ms,
    MAX(max_api_latency_ms)                       AS max_latency_ms,
    SUM(failed_events)                            AS total_failed_events,
    ROUND(AVG(failure_rate_pct), 2)               AS avg_failure_rate_pct
  FROM app_event_complexity
  GROUP BY event_type
  ORDER BY avg_failure_rate_pct DESC
")

write.csv(query101, '/content/drive/MyDrive/DBA/SQL_Queries/query10.1.csv', row.names=FALSE)
print(query101)

                   event_type total_events unlinked_events avg_latency_ms
1              chat_escalated           38               9         478.98
2               payment_retry           69              16         456.01
3                 track_order          138              32         459.13
4                search_route           99              20         441.82
5                 eta_refresh          105              22         463.96
6 delivery_instruction_update           75              16         486.89
7                 chat_opened           88              21         472.87
8              cancel_attempt           28               8         419.29
  max_latency_ms total_failed_events avg_failure_rate_pct
1           1402                  19                49.58
2           1701                  19                27.47
3           1660                   0                 0.00
4           1265                   0                 0.00
5           1285                   0        

## Query 10.2

In [ ]:
%%R

query1012 <- sqldf("
  SELECT
    device_type,
    zone_context,
    SUM(total_events)                             AS total_events,
    SUM(events_without_order)                     AS unlinked_events,
    ROUND(AVG(avg_api_latency_ms), 2)             AS avg_latency_ms,
    SUM(failed_events)                            AS total_failed_events,
    ROUND(AVG(failure_rate_pct), 2)               AS avg_failure_rate_pct
  FROM app_event_complexity
  GROUP BY device_type, zone_context
  ORDER BY avg_failure_rate_pct DESC, avg_latency_ms DESC
  LIMIT 12
")

write.csv(query1012, '/content/drive/MyDrive/DBA/SQL_Queries/query10.12.csv', row.names=FALSE)
print(query1012)

   device_type zone_context total_events unlinked_events avg_latency_ms
1      Android    Riverside           34               9         425.77
2          iOS      Airport           38               8         569.98
3          Web      Central            9               4         525.25
4          Web         West           15               2         423.43
5      Android        North           51              11         414.79
6          Web      Airport           16               4         644.64
7          Web        North           14               1         410.24
8      Android      Airport           33               8         494.12
9          iOS        North           28               5         468.83
10         iOS      Central           31               6         470.88
11     Android         East           46              13         474.28
12     Android         West           47               5         498.74
   total_failed_events avg_failure_rate_pct
1                   

# TABLE 11 — Incident Type Distribution Across Vehicle Fleet

In [ ]:
%%R
# Joining incidents and deliveries using delivery_id
# and joining vehicles and deliveries using vehicle_id
# Connecting vehicle technical profiles with operational incidents to detect maintenance patterns
# This join identifies if specific vehicle types or maintenance states are driving service disruptions
# Will eventually help us identify the bottlenecks

incident_vehicle_profile <- sqldf("
  SELECT
    v.vehicle_type,
    v.maintenance_status,
    i.incident_type,
    i.severity                                                         AS incident_severity,
    COUNT(i.incident_id)                                               AS total_incidents,
    ROUND(AVG(i.resolved_hours), 2)                                    AS avg_resolution_hours,
    SUM(CASE WHEN i.resolution_status = 'Open'      THEN 1 ELSE 0 END) AS open_incidents,
    SUM(CASE WHEN i.resolution_status = 'Escalated' THEN 1 ELSE 0 END) AS escalated_incidents
  FROM incidents i
  JOIN deliveries d ON i.delivery_id = d.delivery_id
  JOIN vehicles   v ON d.vehicle_id  = v.vehicle_id
  GROUP BY v.vehicle_type, v.maintenance_status, i.incident_type, i.severity
  ORDER BY total_incidents DESC
")

print(incident_vehicle_profile)

    vehicle_type maintenance_status    incident_type incident_severity
1       CargoVan             Active   RouteDeviation               Low
2             EV             Active     ProofMissing            Medium
3             EV             Active   RouteDeviation               Low
4       CargoVan             Active     AppSyncError            Medium
5       CargoVan             Active     ProofMissing            Medium
6             EV             Active     BatteryAlert            Medium
7             EV             Active   CustomerNoShow            Medium
8             EV          Scheduled     VehicleFault            Medium
9         Hybrid             Active     VehicleFault              High
10        Hybrid           InRepair     BatteryAlert            Medium
11      CargoVan             Active     ProofMissing               Low
12      CargoVan             Active   RouteDeviation            Medium
13      CargoVan             Active     VehicleFault            Medium
14    

In [ ]:
%%R
# save the table
write.csv(incident_vehicle_profile, '/content/drive/MyDrive/DBA/SQL_Files/incident_vehicle_profile.csv', row.names=FALSE)

In [ ]:
%%R
incident_vehicle_profile <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/incident_vehicle_profile.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

## Query 11.1

In [ ]:
%%R

query111 <- sqldf("
  SELECT
    vehicle_type,
    maintenance_status,
    SUM(total_incidents)                          AS total_incidents,
    ROUND(AVG(avg_resolution_hours), 2)           AS avg_resolution_hours,
    SUM(open_incidents)                           AS total_open,
    SUM(escalated_incidents)                      AS total_escalated,
    ROUND(SUM(open_incidents + escalated_incidents) * 100.0
          / SUM(total_incidents), 2)              AS unresolved_rate_pct
  FROM incident_vehicle_profile
  GROUP BY vehicle_type, maintenance_status
  ORDER BY unresolved_rate_pct DESC, total_incidents DESC
")
write.csv(query111, '/content/drive/MyDrive/DBA/SQL_Queries/query11.1.csv', row.names=FALSE)
print(query111)

   vehicle_type maintenance_status total_incidents avg_resolution_hours
1      CargoVan           InRepair              19                14.40
2      CargoVan          Scheduled               8                11.80
3        Diesel           InRepair              25                12.30
4            EV             Active              61                12.03
5      CargoVan             Active              40                12.05
6        Hybrid             Active              30                10.82
7        Hybrid          Scheduled              13                14.82
8            EV           InRepair              17                12.93
9        Diesel          Scheduled               3                 9.17
10           EV          Scheduled              22                11.79
11       Diesel             Active              19                 9.99
12       Hybrid           InRepair              23                10.67
   total_open total_escalated unresolved_rate_pct
1           7 

## Query 11.2

In [ ]:
%%R

query112 <- sqldf("
  SELECT
    incident_type,
    incident_severity,
    SUM(total_incidents)                          AS total_incidents,
    ROUND(AVG(avg_resolution_hours), 2)           AS avg_resolution_hours,
    SUM(open_incidents)                           AS total_open,
    SUM(escalated_incidents)                      AS total_escalated,
    ROUND(SUM(open_incidents + escalated_incidents) * 100.0
          / SUM(total_incidents), 2)              AS unresolved_rate_pct
  FROM incident_vehicle_profile
  GROUP BY incident_type, incident_severity
  ORDER BY avg_resolution_hours DESC, unresolved_rate_pct DESC
")
write.csv(query112, '/content/drive/MyDrive/DBA/SQL_Queries/query11.2.csv', row.names=FALSE)
print(query112)

      incident_type incident_severity total_incidents avg_resolution_hours
1  TemperatureIssue            Medium              13                17.33
2      AppSyncError               Low               9                16.77
3    RouteDeviation               Low              17                16.28
4      BatteryAlert              High               8                15.78
5    CustomerNoShow            Medium              13                15.50
6    CustomerNoShow              High              11                15.24
7    CustomerNoShow               Low              15                14.85
8  TemperatureIssue               Low               6                14.80
9      VehicleFault          Critical               3                14.57
10     ProofMissing               Low              13                14.35
11   RouteDeviation          Critical               4                14.20
12     BatteryAlert          Critical               2                12.80
13   SafetyNearMiss      

# TABLE 12 — Cross-System Data Fragmentation Evidence

In [ ]:
%%R
# Joining deliveries and order using order_id
# and left joining complaints order and customers using customer_id
# and left joining incidents and deliveries using delivery_id
# Combining logistics, support, and operational datasets to identify failures
# This query isolates failed deliveries to check if they were captured by all internal systems

fragmentation_evidence <- sqldf("
  SELECT
    d.delivery_id,
    d.delivery_status,
    d.manual_route_override_count,
    d.proof_of_completion_missing,
    d.customer_rating_post_delivery,
    o.service_type,
    o.priority_level,
    CASE WHEN c.complaint_id IS NULL THEN 'No Complaint Logged'
         ELSE 'Complaint Exists' END                           AS complaint_presence,
    CASE WHEN i.incident_id  IS NULL THEN 'No Incident Logged'
         ELSE 'Incident Exists' END                            AS incident_presence
  FROM deliveries d
  JOIN orders o          ON d.order_id    = o.order_id
  LEFT JOIN complaints c ON o.customer_id = c.customer_id
  LEFT JOIN incidents  i ON d.delivery_id = i.delivery_id
  WHERE d.delivery_status = 'Failed'
  ORDER BY d.customer_rating_post_delivery ASC
")

print(fragmentation_evidence)

    delivery_id delivery_status manual_route_override_count
1       DL00012          Failed                           3
2       DL00558          Failed                           1
3       DL00536          Failed                           2
4       DL00057          Failed                           0
5       DL00862          Failed                           1
6       DL00862          Failed                           1
7       DL00862          Failed                           1
8       DL00187          Failed                           1
9       DL00839          Failed                           2
10      DL00783          Failed                           2
11      DL00040          Failed                           1
12      DL00694          Failed                           0
13      DL00119          Failed                           3
14      DL00823          Failed                           1
15      DL00559          Failed                           2
16      DL00657          Failed         

In [ ]:
%%R
# save the table
write.csv(fragmentation_evidence, '/content/drive/MyDrive/DBA/SQL_Files/fragmentation_evidence.csv', row.names=FALSE)

In [ ]:
%%R
fragmentation_evidence <- read.csv('/content/drive/MyDrive/DBA/SQL_Files/fragmentation_evidence.csv', stringsAsFactors = FALSE, na.strings = c("", "NA", "N/A"))

## Query 12.1

In [ ]:
%%R

query121 <- sqldf("
  SELECT
    complaint_presence,
    incident_presence,
    COUNT(delivery_id)                            AS total_failed_deliveries,
    ROUND(AVG(customer_rating_post_delivery), 2)  AS avg_rating,
    ROUND(AVG(manual_route_override_count), 2)    AS avg_overrides,
    SUM(proof_of_completion_missing)              AS missing_proof_count
  FROM fragmentation_evidence
  GROUP BY complaint_presence, incident_presence
  ORDER BY total_failed_deliveries DESC
")
write.csv(query121, '/content/drive/MyDrive/DBA/SQL_Queries/query12.1.csv', row.names=FALSE)
print(query121)

   complaint_presence  incident_presence total_failed_deliveries avg_rating
1    Complaint Exists No Incident Logged                      76       3.06
2 No Complaint Logged No Incident Logged                      50       3.04
3 No Complaint Logged    Incident Exists                      21       2.79
4    Complaint Exists    Incident Exists                      20       3.10
  avg_overrides missing_proof_count
1          1.13                  20
2          1.14                   8
3          0.67                   4
4          0.75                   1


## Query 12.2

In [ ]:
%%R

query122 <- sqldf("
  SELECT
    service_type,
    priority_level,
    COUNT(delivery_id)                            AS total_failed_deliveries,
    SUM(CASE WHEN complaint_presence = 'No Complaint Logged'
             AND incident_presence   = 'No Incident Logged'
             THEN 1 ELSE 0 END)                   AS completely_invisible_failures,
    ROUND(SUM(CASE WHEN complaint_presence = 'No Complaint Logged'
                   AND incident_presence   = 'No Incident Logged'
                   THEN 1.0 ELSE 0.0 END) * 100
          / COUNT(delivery_id), 2)                AS invisibility_rate_pct,
    ROUND(AVG(customer_rating_post_delivery), 2)  AS avg_rating,
    SUM(proof_of_completion_missing)              AS missing_proof_count
  FROM fragmentation_evidence
  GROUP BY service_type, priority_level
  ORDER BY invisibility_rate_pct DESC
")
write.csv(query122, '/content/drive/MyDrive/DBA/SQL_Queries/query12.2.csv', row.names=FALSE)
print(query122)

   service_type priority_level total_failed_deliveries
1      Business       Critical                       1
2       Medical       Critical                       1
3        Parcel         Medium                      15
4      Business           High                       6
5        Parcel           High                       6
6      Business            Low                       3
7     Passenger           High                      15
8     Passenger            Low                      17
9        Parcel            Low                       7
10    Passenger         Medium                      18
11       Retail         Medium                      15
12     Business         Medium                      21
13       Retail           High                       9
14       Retail            Low                      10
15      Medical           High                       7
16      Medical         Medium                      10
17      Medical            Low                       5
18    Pass